# 01 - Temporal Feature Engineering

EDA found day of week and month patterns were essentially flat (weak standalone signal). We still engineer these features here, since the Initial Submission commits to it and a weak *individual* signal doesn't mean a feature is useless in combination with others (a model can still pick up interaction effects) - but we shouldn't expect this to move the needle much on its own.


## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

TRAIN_IN = Path('../../../data/processed/train.csv')
VAL_IN = Path('../../../data/processed/val.csv')
TEST_IN = Path('../../../data/processed/test.csv')

TRAIN_OUT = Path('../../../data/processed/features_step1_train.csv')
VAL_OUT = Path('../../../data/processed/features_step1_val.csv')
TEST_OUT = Path('../../../data/processed/features_step1_test.csv')

train_df = pd.read_csv(TRAIN_IN)
val_df = pd.read_csv(VAL_IN)
test_df = pd.read_csv(TEST_IN)

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")


Train: (46026, 21), Val: (7890, 21), Test: (11836, 21)


## 1. Extract temporal features (applied identically to all three splits)

In [2]:
def add_temporal_features(df):
    df = df.copy()
    df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'], errors='coerce')
    df['order_hour'] = df['order date (DateOrders)'].dt.hour
    df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek  # 0=Monday
    df['order_month'] = df['order date (DateOrders)'].dt.month
    df['order_is_weekend'] = (df['order_dayofweek'] >= 5).astype(int)
    df['order_is_holiday_season'] = df['order_month'].isin([11, 12]).astype(int)  # Nov-Dec
    return df

train_df = add_temporal_features(train_df)
val_df = add_temporal_features(val_df)
test_df = add_temporal_features(test_df)

train_df[['order_hour', 'order_dayofweek', 'order_month', 'order_is_weekend', 'order_is_holiday_season']].describe()


,order_hour,order_dayofweek,order_month,order_is_weekend,order_is_holiday_season
count,46026.000000,46026.000000,46026.000000,46026.000000,46026.000000
mean,11.494612,3.003715,6.089363,0.285686,0.152479
std,6.924143,1.997654,3.571665,0.451746,0.359489
min,0.000000,0.000000,1.000000,0.000000,0.000000
25%,5.000000,1.000000,3.000000,0.000000,0.000000
50%,11.000000,3.000000,6.000000,0.000000,0.000000
75%,17.000000,5.000000,9.000000,1.000000,0.000000
max,23.000000,6.000000,12.000000,1.000000,1.000000


## 2. Sanity check against the EDA finding

Confirm the weak signal finding still holds on the train split alone (should roughly match what EDA found on the full dataset).


In [3]:
for col in ['order_dayofweek', 'order_month', 'order_is_weekend', 'order_is_holiday_season']:
    rate = train_df.groupby(col)['Late_delivery_risk'].mean()
    spread = rate.max() - rate.min()
    print(f"{col}: late rate range {rate.min():.3f} to {rate.max():.3f} (spread: {spread:.3f})")


order_dayofweek: late rate range 0.541 to 0.556 (spread: 0.015)
order_month: late rate range 0.535 to 0.559 (spread: 0.024)
order_is_weekend: late rate range 0.547 to 0.552 (spread: 0.005)
order_is_holiday_season: late rate range 0.548 to 0.549 (spread: 0.001)


**What we found:**

**Confirmed, exactly consistent with EDA's "essentially flat" finding on the full dataset:**

| Feature | Late rate range | Spread |
|---|---|---|
| `order_dayofweek` | 0.541 to 0.556 | 1.5 pp |
| `order_month` | 0.535 to 0.559 | 2.4 pp |
| `order_is_weekend` | 0.547 to 0.552 | 0.5 pp |
| `order_is_holiday_season` | 0.548 to 0.549 | 0.1 pp |

All four spreads are under 2.5 percentage points, tiny compared to the 57pp spread found for Shipping Mode or even the 30pp spread for Order Country. This confirms these features are weak on their own, exactly as expected, and none of them shows any surprising split specific noise that would need a second look.

**Decision:** keep all four features in the dataset anyway (per the Initial Submission's plan), since a weak individual signal doesn't rule out the model picking up a useful interaction (for example, holiday season combined with a specific shipping mode). But we should not expect these to show up as important features in the model comparison stage, and if the final feature importance ranking confirms that, it's a clean, expected result worth stating plainly in the report rather than something to explain away.


## 3. Save

In [4]:
train_df.to_csv(TRAIN_OUT, index=False)
val_df.to_csv(VAL_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)
print("Saved step 1 outputs.")


Saved step 1 outputs.
